In [17]:
from snowflake.snowpark import Session

session = Session.builder.configs({
      "account":   "ES10286-ML_ENTERPRISE",
      "user":      "RKIRK",
      "password":  "8d!upvFs2#BDDB5JQ*7",
      "role":      "DEV_RK_FINANCE_SYSADMIN",
      "warehouse": "DAFT_WH",
      "database":  "DEV_RK_FINANCE",
      "schema":    "ACCOUNT_DATA"
  }).create()

### 2. Create a DataFrame from the Snowflake SFO airport weather data

* Define the DataFrame from the SFO_10YR_JSON weather table

In [18]:
dataDF = session.table("TSB_TRANSACTIONS_RAW")


* Examine the table: Show the table structure, row count, and a sample record

Of course here the term *schema* refers to the structure or description of a DataFrame, not a schema object that defines a namespace in Snowflake.

In [19]:
dataDF.schema.fields

[StructField('UPLOAD_ID', LongType(), nullable=False),
 StructField('TRANSACTION_DATE', DateType(), nullable=True),
 StructField('TRANSACTION_TYPE', StringType(10), nullable=True),
 StructField('SORT_CODE', StringType(20), nullable=True),
 StructField('ACCOUNT_NUMBER', StringType(50), nullable=True),
 StructField('DESCRIPTION', StringType(1000), nullable=True),
 StructField('DEBIT_AMOUNT', DecimalType(18, 2), nullable=True),
 StructField('CREDIT_AMOUNT', DecimalType(18, 2), nullable=True),
 StructField('BALANCE', DecimalType(18, 2), nullable=True)]

In [20]:
dataDF.count()


174

In [21]:
dataDF.show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"UPLOAD_ID"  |"TRANSACTION_DATE"  |"TRANSACTION_TYPE"  |"SORT_CODE"  |"ACCOUNT_NUMBER"  |"DESCRIPTION"                                |"DEBIT_AMOUNT"  |"CREDIT_AMOUNT"  |"BALANCE"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|101          |2025-05-30          |FPI                 |30-65-92     |20085168          |Richard Kirk Sent from Revolut               |NULL            |200.00           |145.83     |
|101          |2025-05-30          |DEB                 |30-65-92     |20085168          |AMAZON* 2B28M4CK5 CD 0118                    |15.98           |NULL             |-54.17     |
|101          |2025-05-30          |DD                  |30-65-92     |20085168 

### 3. Flatten the nested structure (two techniques)

#### Option 1: Derive the DataFrame from a SQL SELECT statement
> If the argument to a session.sql() method call is a SELECT statement, then the method defines a DataFrame on the result of the SELECT.

> Note, the SQL statement is placed in triple double quotes in order to accept all quotes and other characters within the statement string.

* Examine the DataFrame

In [22]:
from snowflake.snowpark import Session

session_kana = Session.builder.configs({
      "account":   "KANADEVIAINOVA-EUROPE_DEV",
      "user":      "richard.kirk@kanadevia-inova.com",
      "authenticator": "externalbrowser",
      "role":      "KVI_DCM_DEVELOPER",
      "warehouse": "KVI_OPENFLOW_WH",
      "database":  "BRONZE_RAW",
      "schema":    "IFS_OF"
  }).create()


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/34c07ccb-0a1e-496e-a316-3284040bb923/saml2?SAMLRequest=nZJPj9owEMW%2FSuSekzh%2FoGABKwpaNVqWpQtUbS%2BV40zAwrFT2yG7375OAGl72D30Fk3ezO953kzuXirhnUEbruQURQFGHkimCi4PU7Tf3fsj5BlLZUGFkjBFr2DQ3WxiaCVqMm%2FsUT7DnwaM9dwgaUj3Y4oaLYmihhsiaQWGWEa288cViQNMqDGgrcOha0thuGMdra1JGLZtG7RJoPQhjDHGIR6HTtVJPqE3iPpjRq2VVUyJW8uLe9M7iCjEaYdwCkfYXBu%2FcHlZwUeU%2FCIy5Otut%2FE3T9sd8ua31y2UNE0Fegv6zBnsn1cXA8Y5OFFJCzhzyqU6Ux8arWr47SqBkaotBT0BU1XdWDc9cF9hCUUo1IG7nWXLKapPvIhH8x8P%2BfpRjkuRZU%2FfjnT4k5aro9gfqwear3UrzovDetBWjCHv%2By3huEs4M6aBTHa5WlfC8dDHAz9KdnhA4pTEwyBN0l%2FIW7pcuaS277yZ730EFWdaGVVaJQWX0LtMUoY%2FM5b7mEbgp%2BMh%2BDSJhn4Sj1Kc4jwfx0nYpRejywWR3oie%2Fe9eJuHbKdejXLucsuVGCc5evXulK2rfjzEKor7CC7%2FspQQqysW8KDQY4%2BIUQrULDdS627e6ARTOLtR%2Fr3%2F2Fw%3D%3D&RelayState=ver%3A3-hint%3A788169458368518-ETMsDgAAAZ4fyyDCABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEG4ZIBPzApV51DPyOfQ0rK4AAA

 pip install snowflake-connector-python[secure-local-storage]


In [27]:
addressDF = (
    session_kana.sql("""SELECT DISTINCT
      f.value:"@odata.etag"::STRING        AS odata_etag,
      f.value:SupplierId::STRING           AS supplier_id,
      f.value:AddressId::STRING            AS address_id,
      f.value:Party::STRING                AS party,
      f.value:PartyType::STRING            AS party_type,
      f.value:Address1::STRING             AS address1,
      f.value:Address2::STRING             AS address2,
      f.value:City::STRING                 AS city,
      f.value:Country::STRING              AS country,
      f.value:ZipCode::STRING              AS zip_code,
      f.value:DefaultDomain::BOOLEAN       AS default_domain,
      f.value:OutputMedia::STRING          AS output_media,
      f.value:ValidFrom::DATE              AS valid_from,
      f.value:ValidTo::DATE                AS valid_to,
      f.value:keyref::STRING               AS keyref
  FROM SUPPLIERADDRESS t,
  LATERAL FLATTEN(input => t.RAW:value) f""")
)

* Examine the DataFrame

In [28]:
addressDF.schema.fields

[StructField('ODATA_ETAG', StringType(), nullable=True),
 StructField('SUPPLIER_ID', StringType(), nullable=True),
 StructField('ADDRESS_ID', StringType(), nullable=True),
 StructField('PARTY', StringType(), nullable=True),
 StructField('PARTY_TYPE', StringType(), nullable=True),
 StructField('ADDRESS1', StringType(), nullable=True),
 StructField('ADDRESS2', StringType(), nullable=True),
 StructField('CITY', StringType(), nullable=True),
 StructField('COUNTRY', StringType(), nullable=True),
 StructField('ZIP_CODE', StringType(), nullable=True),
 StructField('DEFAULT_DOMAIN', BooleanType(), nullable=True),
 StructField('OUTPUT_MEDIA', StringType(), nullable=True),
 StructField('VALID_FROM', DateType(), nullable=True),
 StructField('VALID_TO', DateType(), nullable=True),
 StructField('KEYREF', StringType(), nullable=True)]

In [29]:
addressDF.count()

6884

The count is the count of the JSON elements returned by the SELECT LATERAL FLATTEN query, across all the rows in the table.

In [30]:
addressDF.sort(col('SUPPLIER_ID')).show(5)

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ODATA_ETAG"                                        |"SUPPLIER_ID"  |"ADDRESS_ID"  |"PARTY"  |"PARTY_TYPE"  |"ADDRESS1"             |"ADDRESS2"  |"CITY"             |"COUNTRY"  |"ZIP_CODE"  |"DEFAULT_DOMAIN"  |"OUTPUT_MEDIA"  |"VALID_FROM"  |"VALID_TO"  |"KEYREF"                           |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|W/"Vy8iQUFBUmx2QUJUQUFDcEo4QUFTOjIwMjMxMjIwMDc1...  |2001249        |1             |104      |Supplier      |Ul. Z. Neje

#### Option 2: Flatten using DataFrame transformations

> Rather than using SQL, you can transform a DataFrame using methods in the [Snowpark API](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/index.html) itself.

> In the syntax below, multilple DataFrame transformations are chained together, so that each line defines a new DataFrame derived from the line just above.

> For a definitive list of the methods available on DataFrames, see the [DataFrame class](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/dataframe.html) in the API documentation.

In [10]:
addressDF_raw = session_kana.table("SUPPLIERADDRESS")

In [11]:
addressDF_raw.show(1)

------------------------------------------------------------------------------------------------
|"RAW"                                               |"INGESTION_TIMESTAMP"      |"LOAD_MODE"  |
------------------------------------------------------------------------------------------------
|{                                                   |2026-05-05 09:19:51+00:00  |FULL         |
|  "@odata.context": "https://kvinova-uat.ifs.cl...  |                           |             |
|  "value": [                                        |                           |             |
|    {                                               |                           |             |
|      "@odata.etag": "W/\"Vy8iQUFBUmx2QUJUQUFMT...  |                           |             |
|      "Address": "Arran Moore House\r\n - ML7 5...  |                           |             |
|      "Address1": "Arran Moore House",              |                           |             |
|      "Address2": null,      

In [12]:
from snowflake.snowpark.functions import col, lit
from snowflake.snowpark.types import StringType, BooleanType, DateType

addressDF2 = (
    addressDF_raw.
    join_table_function('flatten', col('RAW'), lit('value')).
    withColumn('odata_etag',     col('VALUE')['"@odata.etag"'].cast(StringType())).
    withColumn('supplier_id',    col('VALUE')['SupplierId'].cast(StringType())).
    withColumn('address_id',     col('VALUE')['AddressId'].cast(StringType())).
    withColumn('party',          col('VALUE')['Party'].cast(StringType())).
    withColumn('party_type',     col('VALUE')['PartyType'].cast(StringType())).
    withColumn('address1',       col('VALUE')['Address1'].cast(StringType())).
    withColumn('address2',       col('VALUE')['Address2'].cast(StringType())).
    withColumn('city',           col('VALUE')['City'].cast(StringType())).
    withColumn('country',        col('VALUE')['Country'].cast(StringType())).
    withColumn('zip_code',       col('VALUE')['ZipCode'].cast(StringType())).
    withColumn('default_domain', col('VALUE')['DefaultDomain'].cast(BooleanType())).
    withColumn('output_media',   col('VALUE')['OutputMedia'].cast(StringType())).
    withColumn('valid_from',     col('VALUE')['ValidFrom'].cast(DateType())).
    withColumn('valid_to',       col('VALUE')['ValidTo'].cast(DateType())).
    withColumn('keyref',         col('VALUE')['keyref'].cast(StringType())).
    select('odata_etag', 'supplier_id', 'address_id', 'party', 'party_type',
           'address1', 'address2', 'city', 'country', 'zip_code',
           'default_domain', 'output_media', 'valid_from', 'valid_to', 'keyref').
    distinct()
)

* Examine the DataFrame

In [13]:
addressDF2.schema.fields

[StructField('ODATA_ETAG', StringType(), nullable=True),
 StructField('SUPPLIER_ID', StringType(), nullable=True),
 StructField('ADDRESS_ID', StringType(), nullable=True),
 StructField('PARTY', StringType(), nullable=True),
 StructField('PARTY_TYPE', StringType(), nullable=True),
 StructField('ADDRESS1', StringType(), nullable=True),
 StructField('ADDRESS2', StringType(), nullable=True),
 StructField('CITY', StringType(), nullable=True),
 StructField('COUNTRY', StringType(), nullable=True),
 StructField('ZIP_CODE', StringType(), nullable=True),
 StructField('DEFAULT_DOMAIN', BooleanType(), nullable=True),
 StructField('OUTPUT_MEDIA', StringType(), nullable=True),
 StructField('VALID_FROM', DateType(), nullable=True),
 StructField('VALID_TO', DateType(), nullable=True),
 StructField('KEYREF', StringType(), nullable=True)]

In [14]:
addressDF2.count()

6884

In [15]:
addressDF2.sort(col('supplier_id')).show(5)

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ODATA_ETAG"  |"SUPPLIER_ID"  |"ADDRESS_ID"  |"PARTY"  |"PARTY_TYPE"  |"ADDRESS1"             |"ADDRESS2"  |"CITY"             |"COUNTRY"  |"ZIP_CODE"  |"DEFAULT_DOMAIN"  |"OUTPUT_MEDIA"  |"VALID_FROM"  |"VALID_TO"  |"KEYREF"                           |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|NULL          |2001249        |1             |104      |Supplier      |Ul. Z. Nejedlého 13    |NULL        |Levice             |SK         |93401       |True              |Printout        |NULL          |NULL        |ADDRESS_ID=1^SUPP

### 4. Save flattened data (two ways)

* (1) As a Snowflake **view**

In [16]:
view_to_create = 'BRONZE_RAW.IFS_OF.SUPPLIERADDRESS_DAFTVIEW_VW'
addressDF2.createOrReplaceView(view_to_create)

[Row(status='View SUPPLIERADDRESS_DAFTVIEW_VW successfully created.')]

creates a view with flatten SQL logic;

```sql
create or replace view BRONZE_RAW.IFS_OF.SUPPLIERADDRESS_DAFTVIEW_VW(
	ODATA_ETAG,
	SUPPLIER_ID,
	ADDRESS_ID,
	PARTY,
	PARTY_TYPE,
	ADDRESS1,
	ADDRESS2,
	CITY,
	COUNTRY,
	ZIP_CODE,
	DEFAULT_DOMAIN,
	OUTPUT_MEDIA,
	VALID_FROM,
	VALID_TO,
	KEYREF
) as  SELECT  * 
 FROM (
 SELECT 
    "ODATA_ETAG", 
    "SUPPLIER_ID", 
    "ADDRESS_ID", 
    "PARTY", 
    "PARTY_TYPE", 
    "ADDRESS1", 
    "ADDRESS2", 
    "CITY", 
    "COUNTRY", 
    "ZIP_CODE", 
    "DEFAULT_DOMAIN", 
    "OUTPUT_MEDIA", 
    "VALID_FROM", 
    "VALID_TO", 
    "KEYREF"
 FROM (
 SELECT 
     CAST ("VALUE"['"@odata.etag"'] AS STRING) AS "ODATA_ETAG", 
     CAST ("VALUE"['SupplierId'] AS STRING) AS "SUPPLIER_ID", 
     CAST ("VALUE"['AddressId'] AS STRING) AS "ADDRESS_ID", 
     CAST ("VALUE"['Party'] AS STRING) AS "PARTY", 
     CAST ("VALUE"['PartyType'] AS STRING) AS "PARTY_TYPE", 
     CAST ("VALUE"['Address1'] AS STRING) AS "ADDRESS1", 
     CAST ("VALUE"['Address2'] AS STRING) AS "ADDRESS2", 
     CAST ("VALUE"['City'] AS STRING) AS "CITY", 
     CAST ("VALUE"['Country'] AS STRING) AS "COUNTRY", 
     CAST ("VALUE"['ZipCode'] AS STRING) AS "ZIP_CODE", 
     CAST ("VALUE"['DefaultDomain'] AS BOOLEAN) AS "DEFAULT_DOMAIN", 
     CAST ("VALUE"['OutputMedia'] AS STRING) AS "OUTPUT_MEDIA", 
     CAST ("VALUE"['ValidFrom'] AS DATE) AS "VALID_FROM", 
     CAST ("VALUE"['ValidTo'] AS DATE) AS "VALID_TO", 
     CAST ("VALUE"['keyref'] AS STRING) AS "KEYREF"
 FROM (
 SELECT 
    T_LEFT.*, 
    T_RIGHT.*
 FROM (
 SELECT  *  FROM SUPPLIERADDRESS
) AS T_LEFT
 JOIN 
 TABLE (flatten("RAW", 'value') ) AS T_RIGHT
)
)
 GROUP BY 
    "ODATA_ETAG", 
    "SUPPLIER_ID", 
    "ADDRESS_ID", 
    "PARTY", 
    "PARTY_TYPE", 
    "ADDRESS1", 
    "ADDRESS2", 
    "CITY", 
    "COUNTRY", 
    "ZIP_CODE", 
    "DEFAULT_DOMAIN", 
    "OUTPUT_MEDIA", 
    "VALID_FROM", 
    "VALID_TO", 
    "KEYREF"
);
``` 

* (2) As a Snowflake **table**: Materialize the content of the DataFrame and save as a table in Snowflake.

> The write() method on a DataFrame returns a [DataFrameWriter](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrameWriter.html#snowflake.snowpark.DataFrameWriter) object.

> Prior to writing a table to Snowflake, you can set the **mode** for saving data to 'append', 'overwrite', 'errorifexists', or 'ignore'.

In [31]:
table_to_save = 'BRONZE_RAW.IFS_OF.SUPPLIERADDRESS_DAFT_FLATTENED'

(addressDF2.
 write.
 mode('overwrite').
 save_as_table(table_to_save)
)